# Medallion Architecture — Bronze → Silver → Gold

Recap of the Medallion Architecture in the context of **transformation pipelines**. We’ll explore data flow design principles, modularity, and separation of processing logic — including practical SCD demos.

| Training Block | Duration | Type |
|---|---|---|
| Medallion Architecture Recap | 30 min | Demo + Discussion |

**Prerequisites:** Databricks Fundamentals training (familiar with Bronze/Silver/Gold concepts, Delta Lake basics).

## Learning Objectives

After completing this module you will be able to:

- **Recap** the Medallion Architecture layers (Bronze → Silver → Gold) in the context of production pipelines
- **Implement** SCD Type 1 (overwrite) using traditional `MERGE INTO`
- **Implement** SCD Type 2 (history tracking) using traditional `MERGE INTO`
- **Compare** the traditional approach with Lakeflow `AUTO CDC` declarations
- **Design** data flows between Silver and Gold layers with clear responsibilities
- **Apply** the modularity principle: load → transform → save

## Setup

---

In [ ]:
%run ../../setup/00_setup

## Slowly Changing Dimensions (SCD)

**Slowly Changing Dimension (SCD)** — strategies for handling changes in dimensional data over time.

| Type | Strategy | Result |
|-----|-----------|----------|
| **SCD Type 0** | Retain original | Always Warsaw |
| **SCD Type 1** | Overwrite | Only Krakow |
| **SCD Type 2** | Track history | Both records with dates |
| **SCD Type 3** | Add column | `current_city=Krakow`, `previous_city=Warsaw` |

In this demo we’ll focus on **Type 1** and **Type 2** — the two most common patterns in data engineering.

---

## SCD Type 1 — Overwrite (Traditional Approach)

- **No history** — old values are overwritten
- **Use cases:** Error corrections, non-historical attributes (e.g., email change)
- **Implementation:** `MERGE INTO ... WHEN MATCHED THEN UPDATE`

---

In [ ]:
# SCD Type 1 — Create dimension table
SCD1_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd1_demo"

spark.sql(f"DROP TABLE IF EXISTS {SCD1_TABLE}")
spark.sql(f"""
CREATE TABLE {SCD1_TABLE} (
    customer_id   STRING,
    name          STRING,
    city          STRING,
    email         STRING,
    updated_at    TIMESTAMP
) USING DELTA
""")

# Insert initial dimension data
spark.sql(f"""
INSERT INTO {SCD1_TABLE} VALUES
    ('C001', 'Anna Kowalska',  'Warsaw',  'anna@example.com',   current_timestamp()),
    ('C002', 'Jan Nowak',      'Gdansk',  'jan@example.com',    current_timestamp()),
    ('C003', 'Maria Wisniewska','Poznan', 'maria@example.com',  current_timestamp())
""")

print(f"Dimension table created: {SCD1_TABLE}")
display(spark.table(SCD1_TABLE))

**Scenario:** Anna moved from Warsaw → Krakow, and a new customer Piotr arrives.

With SCD Type 1, Anna’s old city (Warsaw) will be **overwritten** — no history preserved.

In [ ]:
# Incoming changes (e.g., from a staging table or CDC feed)
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW customer_updates AS
SELECT * FROM VALUES
    ('C001', 'Anna Kowalska',   'Krakow', 'anna.new@example.com'),  -- UPDATE: city + email changed
    ('C004', 'Piotr Zielinski', 'Wroclaw', 'piotr@example.com')     -- INSERT: new customer
AS t(customer_id, name, city, email)
""")

display(spark.sql("SELECT * FROM customer_updates"))

In [ ]:
# SCD Type 1: MERGE INTO — overwrite matched rows, insert new ones
spark.sql(f"""
MERGE INTO {SCD1_TABLE} AS target
USING customer_updates AS source
ON target.customer_id = source.customer_id

WHEN MATCHED THEN UPDATE SET
    target.name       = source.name,
    target.city       = source.city,
    target.email      = source.email,
    target.updated_at = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
    customer_id, name, city, email, updated_at
) VALUES (
    source.customer_id, source.name, source.city, source.email, current_timestamp()
)
""")

print("SCD Type 1 MERGE complete")
display(spark.table(SCD1_TABLE).orderBy("customer_id"))

> **Observation:** Anna’s city is now `Krakow` — Warsaw is **gone forever**. Piotr was inserted as a new row.
>
> SCD Type 1 is simple but **loses history**. If you need to know that Anna was in Warsaw, you need Type 2.

---

## SCD Type 2 — Track History (Traditional Approach)

- **Full history** of changes with `valid_from`, `valid_to`, `is_current`
- **Use cases:** Audit trails, historical analysis, regulatory compliance
- **Current record:** `is_current = true` (or `valid_to IS NULL`)
- **Implementation:** Multi-step MERGE — close old record + insert new one

---

In [ ]:
# SCD Type 2 — Create dimension table with history columns
SCD2_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2_demo"

spark.sql(f"DROP TABLE IF EXISTS {SCD2_TABLE}")
spark.sql(f"""
CREATE TABLE {SCD2_TABLE} (
    customer_id   STRING,
    name          STRING,
    city          STRING,
    email         STRING,
    valid_from    TIMESTAMP,
    valid_to      TIMESTAMP,
    is_current    BOOLEAN
) USING DELTA
""")

# Insert initial dimension data (all records are current)
spark.sql(f"""
INSERT INTO {SCD2_TABLE} VALUES
    ('C001', 'Anna Kowalska',  'Warsaw',  'anna@example.com',   current_timestamp(), NULL, true),
    ('C002', 'Jan Nowak',      'Gdansk',  'jan@example.com',    current_timestamp(), NULL, true),
    ('C003', 'Maria Wisniewska','Poznan', 'maria@example.com',  current_timestamp(), NULL, true)
""")

print(f"SCD2 dimension table created: {SCD2_TABLE}")
display(spark.table(SCD2_TABLE))

**Same scenario:** Anna moved from Warsaw → Krakow.

With SCD Type 2, we need to:
1. **Close** the old record: set `valid_to = now`, `is_current = false`
2. **Insert** a new record: `city = Krakow`, `valid_from = now`, `is_current = true`

This requires a **two-step approach** — standard `MERGE` can’t do both in one statement.

In [ ]:
# SCD Type 2 — Step 1: Close old records for changed customers
spark.sql(f"""
MERGE INTO {SCD2_TABLE} AS target
USING customer_updates AS source
ON target.customer_id = source.customer_id
   AND target.is_current = true

WHEN MATCHED AND (
    target.city <> source.city
    OR target.email <> source.email
    OR target.name <> source.name
) THEN UPDATE SET
    target.valid_to   = current_timestamp(),
    target.is_current = false
""")

print("Step 1 complete: old records closed")
display(spark.table(SCD2_TABLE).orderBy("customer_id", "valid_from"))

In [ ]:
# SCD Type 2 — Step 2: Insert new versions of changed rows + brand new customers
spark.sql(f"""
INSERT INTO {SCD2_TABLE}
SELECT
    source.customer_id,
    source.name,
    source.city,
    source.email,
    current_timestamp() AS valid_from,
    NULL AS valid_to,
    true AS is_current
FROM customer_updates AS source
LEFT JOIN {SCD2_TABLE} AS target
    ON source.customer_id = target.customer_id
    AND target.is_current = true
WHERE target.customer_id IS NULL  -- either new customer or just-closed record
""")

print("Step 2 complete: new versions inserted")
display(spark.table(SCD2_TABLE).orderBy("customer_id", "valid_from"))

> **Observation:** Anna now has **2 rows** — Warsaw (closed: `is_current=false`) and Krakow (active: `is_current=true`).
>
> SCD Type 2 preserves full history but requires **more complex SQL** (2-step MERGE) and **more storage**.

---

In [ ]:
# SCD2 Query Patterns

# Current state only (equivalent to SCD1)
print("=== Current state ===")
display(spark.sql(f"SELECT * FROM {SCD2_TABLE} WHERE is_current = true ORDER BY customer_id"))

# Full history for a specific customer
print("=== Anna's full history ===")
display(spark.sql(f"SELECT * FROM {SCD2_TABLE} WHERE customer_id = 'C001' ORDER BY valid_from"))

## The Lakeflow Way — AUTO CDC

The traditional MERGE approach works but is **verbose and error-prone**.

Lakeflow Declarative Pipelines provide `AUTO CDC` — a single declaration that handles both SCD Type 1 and Type 2 automatically.

---

### AUTO CDC — SCD Type 1

```sql
CREATE FLOW silver_customers_scd1
AS AUTO CDC INTO silver_customers
FROM bronze_customers
KEYS (customer_id)
SEQUENCE BY updated_at
STORED AS SCD TYPE 1;
```

That’s it — **6 lines** instead of a full MERGE statement. Lakeflow handles:
- Deduplication by `KEYS`
- Ordering by `SEQUENCE BY`
- Overwrite semantics (SCD1)

### AUTO CDC — SCD Type 2

```sql
CREATE OR REFRESH STREAMING TABLE silver_customers (
    customer_id   STRING,
    name          STRING,
    city          STRING,
    email         STRING,
    -- SCD2 columns added automatically:
    __START_AT    TIMESTAMP,
    __END_AT      TIMESTAMP
);

CREATE FLOW silver_customers_scd2
AS AUTO CDC INTO silver_customers
FROM STREAM bronze_customers
KEYS (customer_id)
SEQUENCE BY updated_at
STORED AS SCD TYPE 2;
```

Same result as our 2-step MERGE — but:
- `__START_AT` / `__END_AT` managed automatically
- No manual "close old + insert new" logic
- Built-in idempotency and exactly-once guarantees

> We’ll use `AUTO CDC` in practice in **03 — Lakeflow Pipelines Demo**.

### Comparison: Traditional vs Lakeflow

| Aspect | Traditional MERGE | Lakeflow AUTO CDC |
|---|---|---|
| **SCD1 complexity** | Single MERGE statement | Single `STORED AS SCD TYPE 1` declaration |
| **SCD2 complexity** | 2-step MERGE (close + insert) | Single `STORED AS SCD TYPE 2` declaration |
| **History columns** | Manual (`valid_from`, `valid_to`, `is_current`) | Automatic (`__START_AT`, `__END_AT`) |
| **Deduplication** | Manual in MERGE condition | Automatic via `KEYS` + `SEQUENCE BY` |
| **Idempotency** | Must be designed carefully | Built-in |
| **Delete handling** | Additional MERGE clause | `APPLY AS DELETE WHEN` |
| **Where to use** | Any Spark/SQL environment | Inside Lakeflow Declarative Pipeline |

---

## Cleanup

In [ ]:
# Remove SCD demo tables
spark.sql(f"DROP TABLE IF EXISTS {SCD1_TABLE}")
spark.sql(f"DROP TABLE IF EXISTS {SCD2_TABLE}")
print("SCD demo tables dropped")

---

## Designing Data Flows in the Medallion Architecture

In the Fundamentals training, you built a basic Bronze → Silver → Gold pipeline. Now let's look at **production design principles**.

---

### Bronze Layer — Raw Ingestion

**Principle:** Keep raw data as-is. Add metadata, never transform.

| Aspect | Guideline |
|--------|-----------|
| Schema | Cast everything to STRING or keep original types |
| Metadata | Add `_metadata.file_path`, `ingestion_ts`, `source_system` |
| Deduplication | None — keep all records (handle in Silver) |
| Quality | None — no validation at this layer |
| Write mode | **Append only** — never overwrite |

```sql
CREATE OR REFRESH STREAMING TABLE bronze_orders
AS SELECT
 *,
 _metadata.file_path AS source_file,
 current_timestamp() AS ingestion_ts
FROM STREAM read_files('${source_path}/orders/', format => 'json');
```

### Silver Layer — Cleansed & Conformed

**Principle:** Apply business rules, validate quality, conform schema.

| Aspect | Guideline |
|--------|-----------|
| Schema | Strong types (DATE, DECIMAL, INT) |
| Deduplication | By business key + sequence column |
| Quality | Expectations / constraints (null checks, range checks) |
| SCD | Apply SCD Type 1 or 2 for dimensions |
| Write mode | **MERGE** or **AUTO CDC** |

**Key transformations at Silver:**
- Type casting and standardization
- Null handling (coalesce, default values)
- Deduplication (by business key)
- Enrichment (lookups from reference tables)
- SCD management for dimensions

### Gold Layer — Business-Ready

**Principle:** Domain-specific aggregations and star schema tables optimized for consumption.

| Aspect | Guideline |
|--------|-----------|
| Schema | Denormalized / star schema |
| Transformations | Aggregations, window functions, KPIs |
| Consumers | BI tools, dashboards, ML features |
| Write mode | **Overwrite** (MATERIALIZED VIEW) or **Append** |

**Examples:**
- `fact_sales` — joins orders + customers + products
- `agg_daily_revenue` — daily revenue by category
- `dim_customer` — SCD2 snapshot (current records only)

---

## Modularity: Load → Transform → Save

**Anti-pattern:** One giant notebook that reads, transforms, and writes everything.

**Best practice:** Separate concerns into modular, testable steps:

```
┌─────────────┐    ┌──────────────────┐    ┌─────────────┐
│   LOAD      │───▶│   TRANSFORM      │───▶│   SAVE      │
│             │    │                  │    │             │
│ • read_files│    │ • cast types     │    │ • Delta     │
│ • Auto Loader│   │ • validate       │    │ • managed   │
│ • COPY INTO │    │ • deduplicate    │    │   table     │
│ • cloudFiles│    │ • enrich         │    │ • checkpoint│
└─────────────┘    └──────────────────┘    └─────────────┘
```

**Why modular?**
- Each step can be tested independently
- Reusable across pipelines (same transform logic, different sources)
- Easier debugging (isolate where data quality breaks)
- Clear ownership (data engineering vs. analytics teams)

**In Lakeflow:** This modularity is built-in. Each `CREATE STREAMING TABLE` or `CREATE MATERIALIZED VIEW` is a separate, declarative step with automatic dependency management.

---

## Putting It All Together

In the following notebooks today, we will:

1. **02 — Batch & Stream Ingestion:** Load data into Bronze using COPY INTO and Auto Loader
2. **03 — Lakeflow Pipelines:** Build a declarative Bronze → Silver → Gold pipeline with expectations
3. **04 — Orchestration:** Schedule and monitor pipeline execution with Databricks Workflows

Each step builds on the Medallion principles covered here.

---

---

**[ README](../../../README.md)** | [02 — Batch & Stream Ingestion](02_batch_stream_ingestion_demo.ipynb) →
